# 제로베이스 데이터 취업 스쿨 SQL 과제 1
- 스타벅스 이디야 데이터 분석

### 8문제 총 100점

- 1번 5점
- 2번 5점
- 3번 10점
- 4번 5점
- 5번 15점
- 6번 15점
- 7번 40점
- 8번 5점

### 1 ~ 8번 모두 본 노트북 파일에 답안 작성해서 제출해주세요 :)

---

문제 1.

AWS RDS (MySQL) 에 프로젝트 관련 Database 를 생성하고, 접근 가능한 사용자 계정을 생성하세요.

- Database Name : oneday
- User Name / Password : oneday / 1234

In [2]:
import mysql.connector

In [4]:
remote = mysql.connector.connect(
    host = 'database-1.cjnz1ulpkijp.ap-southeast-2.rds.amazonaws.com',
    port=3306,
    user='admin',
    password="mooltisue305!"
)

cursor = remote.cursor(buffered=True)

sql = "CREATE DATABASE oneday"
cursor.execute(sql)

cursor.execute("CREATE USER 'oneday'@'%' identified by '1234'")
cursor.execute("GRANT ALL ON oneday.* to 'oneday'@'%'")

DatabaseError: 1007 (HY000): Can't create database 'oneday'; database exists

제출 1.
- Database 생성문 조회 결과 : SHOW CREATE DATABASE oneday;
- 사용자 권한 확인 결과 : SHOW GRANT FOR ‘oneday’@‘%’

In [5]:
cursor.execute("show create database oneday")
cursor.execute("show grants for 'oneday'@'%'")

문제 2.

스타벅스 이디야 데이터를 저장할 테이블을 다음의 구조로 생성하세요. (PDF 파일 참고)

In [6]:
cursor.execute("use oneday")

cur = remote.cursor()
cur.execute("create table COFFEE_BRAND (id int not null auto_increment primary key, name varchar(16))")
cur.execute("create table COFFEE_STORE (id int not null auto_increment Primary Key, brand int not null, name varchar(32) not null, gu_name varchar(5) not null, address varchar(128) not null,lat decimal(16,14) not null, lng decimal(17,14) not null, foreign key (brand) references COFFEE_BRAND(id))")


문제 3.

Python 코드로 COFFEE_BRAND 데이터를 다음과 같이 입력하고 확인하세요. (PDF 파일 참고)

In [13]:
cursor1 = remote.cursor(buffered=True)
cur.execute("insert into COFFEE_BRAND values (1,'STARBUCKS')")
cur.execute("insert into COFFEE_BRAND values (2,'EDIYA')")
remote.commit()
remote.close()

제출 2.
- Table 생성 결과 : Desc COFFEE_BRAND; Desc COFFEE_STORE;

제출 3.
- COFFEE_BRAND 조회 결과 : SELECT * FROM COFFEE_BRAND;

문제 4.

스타벅스 페이지에 접근하는 코드에서 팝업창이 없는 경우, 팝업창을 닫는 코드에서 에러가 발생합니다. 예외처리 해서 에러
메시지를 출력하고 실행이 중단되지 않도록 수정해주세요.

In [31]:
import warnings
import time
from selenium.webdriver.common.by import By
from selenium import webdriver
from bs4 import BeautifulSoup
from tqdm import tqdm_notebook
warnings.simplefilter(action = 'ignore')

In [132]:
driver = webdriver.Chrome()
driver.get("https://www.starbucks.co.kr/store/store_map.do")

# 팝업창 닫기
popup = driver.window_handles
print(popup)


for i in popup:
    if i != popup[0]:
        driver.switch_to.window(i)
        driver.close()
    

#지역 선택 클릭
driver.find_element(By.XPATH, '//*[@id="container"]/div/form/fieldset/div/section/article[1]/article/header[2]/h3/a').click()
#서울 선택
driver.find_element(By.XPATH, '//*[@id="container"]/div/form/fieldset/div/section/article[1]/article/article[2]/div[1]/div[2]/ul/li[1]/a').click()

time.sleep(3)

#서울 전체 선택
driver.find_element(By.XPATH, '//*[@id="mCSB_2_container"]/ul/li[1]/a').click()

#Beautifulsoup 을 이용하여 html불러오기
html = driver.page_source
dom = BeautifulSoup(html, "html.parser")

test_soup = dom.find_all(style="background:#fff",class_="quickResultLstCon")
time.sleep(8)

len(test_soup)

['88E75F8C35D9AD47253D5789CAB91103']


10

문제 5.

Python 코드로 스타벅스 페이지에서 데이터를 가져올때, COFFEE_STORE 테이블에 바로 입력하도록 수정하세요.

- 데이터 세트: 매장 이름, 매장이 위치한 구 이름, 매장 주소, 위도, 경도
- 필요한 데이터를 한세트씩 가져와서 COFFEE_STORE 테이블에 각각INSERT 하도록 합니다.
- 입력된 데이터의 총 갯수를 쿼리하여 결과를 확인합니다.
- 입력된 데이터 상위 10개를 쿼리하여 결과를 확인합니다.

In [169]:
import mysql.connector

conn = mysql.connector.connect(
    host = 'database-1.cjnz1ulpkijp.ap-southeast-2.rds.amazonaws.com',
    port=3306,
    user='oneday',
    password="1234",
    database = 'oneday'
)

cs = conn.cursor(buffered=True)

sql = "INSERT INTO COFFEE_STORE VALUES (%s,%s,%s,%s,%s,%s)"

for i in tqdm_notebook(range(1, len(test_soup))):
    name = test_soup[i]["data-name"].strip(),
    lat = test_soup[i]["data-lat"].strip(),
    lng = test_soup[i]["data-long"].strip(),
    address = test_soup[i].find('p').text.strip().replace('1522-3232',"")
    gu_name = address.split()[1]

    cs.executemany(sql,(name, gu_name, address, lat, lng))
    conn.commit()

driver.close()

  0%|          | 0/9 [00:00<?, ?it/s]

연수구
연수구
연수구
미추홀구
미추홀구
미추홀구
연수구
연수구
미추홀구


WebDriverException: Message: disconnected: not connected to DevTools
  (failed to check if window was closed: disconnected: not connected to DevTools)
  (Session info: chrome=119.0.6045.105)
Stacktrace:
	GetHandleVerifier [0x00007FF6AD9582B2+55298]
	(No symbol) [0x00007FF6AD8C5E02]
	(No symbol) [0x00007FF6AD7805AB]
	(No symbol) [0x00007FF6AD76D1AA]
	(No symbol) [0x00007FF6AD76D9CE]
	(No symbol) [0x00007FF6AD780AF8]
	(No symbol) [0x00007FF6AD75FB90]
	(No symbol) [0x00007FF6AD7EC714]
	(No symbol) [0x00007FF6AD7E2070]
	(No symbol) [0x00007FF6AD7B670A]
	(No symbol) [0x00007FF6AD7B7964]
	GetHandleVerifier [0x00007FF6ADCD0AAB+3694587]
	GetHandleVerifier [0x00007FF6ADD2728E+4048862]
	GetHandleVerifier [0x00007FF6ADD1F173+4015811]
	GetHandleVerifier [0x00007FF6AD9F47D6+695590]
	(No symbol) [0x00007FF6AD8D0CE8]
	(No symbol) [0x00007FF6AD8CCF34]
	(No symbol) [0x00007FF6AD8CD062]
	(No symbol) [0x00007FF6AD8BD3A3]
	BaseThreadInitThunk [0x00007FFDBCDC7344+20]
	RtlUserThreadStart [0x00007FFDBDFC26B1

제출 4.
- 팝업 예외처리 코드 & 실행 결과 (ipynb)

제출 5.
- 스타벅스 데이터 관련 코드 & 실행 결과 (ipynb)

문제 6.

Python 코드로 이디야 페이지에서 데이터를 가져올때, COFFEE_STORE 테이블에 바로 입력하도록 수정하세요.

- 데이터 세트 : 매장 이름, 매장이 위치한 구 이름, 매장 주소, 위도, 경도
- 이디야 페이지에서 검색에 사용할 구 이름은 COFFEE_STORE 에서 중복을 제거하는 쿼리를 사용하여 가져와서 {‘서울 ‘ + 구이름} 형식으로 변환하여 사용하도록 합니다.
- 필요한 데이터를 한 세트씩 가져와서 COFFEE_STORE 테이블에 각각 INSERT 하도록 합니다.
- (주의) COFFEE_STORE 테이블에 입력할 구 이름은 {‘서울 ‘} 이 제거된 구 이름입니다.
- 입력된 데이터의 총 갯수를 쿼리하여 결과를 확인합니다.
- 입력된 데이터 상위 10개를 쿼리하여 결과를 확인합니다.

In [172]:
# 이디야 페이지 접근
url = "https://www.ediya.com/contents/find_store.html#c"
driver = webdriver.Chrome()
driver.get(url)

# 주소 클릭
xpath_address = '//*[@id="contentWrap"]/div[3]/div/div[1]/ul/li[2]/a'
some_tag_address = driver.find_element(By.XPATH, xpath_address)
some_tag_address.click()

#검색어 입력
write = driver.find_element(By.XPATH, '//*[@id="keyword"]')



제출 6.
- 이디야 데이터 관련 코드 & 실행 결과 (ipynb)

문제 7.

Python 코드에서 다음의 데이터를 쿼리를 사용하여 조회하세요.

- 스타벅스 매장 주요 분포 지역 (매장수가 많은 상위 5개 구이름, 매장 개수 출력)
- 이디야 매장 주요 분포 지역 (매장수가 많은 상위 5개 구이름, 매장 개수 출력)
- 구별 브랜드 각각의 매장 개수 조회 (구이름, 브랜드이름, 매장 개수 출력)
- 구별 브랜드 각각의 매장 개수 조회 (구이름, 스타벅스 매장 개수, 이디야 매장 개수 출력)

제출 7.
- 관련 코드 & 실행 결과 (ipynb)

문제 8.

시각화 프로젝트를 위하여 다음의 규칙으로 쿼리하여 CSV 파일로 저장합니다. (Python 코드로 작업)

- 전체 데이터를 가져오는데, 각 스타벅스 매장별로 이디야 전체 매장정보가 매칭되어 있어야 합니다. (정렬 : s_id, e_id 순)
- 다음의 형식으로 저장되어야 합니다. (브랜드 이름, 칼럼 명 주의)
- 데이터 프레임 출력을 해주세요. 데이터 프레임 미출력시 감점입니다.

제출 8.
- 시각화 프로젝트 관련 코드 (ipynb), 결과 파일 (csv)

---